# Визуализация и разведовательный анализ временных рядов

## Материалы к работе

### Атрибуты

<!-- [<img src="https://camo.githubusercontent.com/eff96fda6b2e0fff8cdf2978f89d61aa434bb98c00453ae23dd0aab8d1451633/68747470733a2f2f636f6c61622e72657365617263682e676f6f676c652e636f6d2f6173736574732f636f6c61622d62616467652e737667" width='100px'>]( -->

#### [Coloab](https://colab.research.google.com/github/MVRonkin/TimeSeriesCourse/blob/main/Last/WS/01-EDA_VIS.ipynb)

#### утилиты

##### Скачиваем если их нет

In [3]:
import os
UPDATE_UTILITS = True
if UPDATE_UTILITS:
    !rm -rf utilits4tsc temp_repo
    !git clone --depth 1 https://github.com/MVRonkin/utilits4tsc.git utilits4tsc
    

fatal: destination path 'temp_repo' already exists and is not an empty directory.
"mv" ­Ґ пў«пҐвбп ў­гваҐ­­Ґ© Ё«Ё ў­Ґи­Ґ©
Є®¬ ­¤®©, ЁбЇ®«­пҐ¬®© Їа®Ја ¬¬®© Ё«Ё Ї ЄҐв­л¬ д ©«®¬.


##### Плот по гост

In [4]:
from utilits4tsc import *
plt_style_GOST()

##### Содержание

In [5]:
from IPython.display import display, Markdown
display(Markdown(generate_toc(max_lvl=2)))

## Содержание
* [Визуализация и разведовательный анализ временных рядов](#визуализация-и-разведовательный-анализ-временных-рядов)
  * [Материалы к работе](#материалы-к-работе)
  * [Загрузка данных в Pandas](#загрузка-данных-в-pandas)
  * [Первичный анализ данных](#первичный-анализ-данных)
  * [Визуализация временного ряда](#визуализация-временного-ряда)
  * [Оценка сезонных составляющих](#оценка-сезонных-составляющих)
  * [Анализ многопеременного ВР](#анализ-многопеременного-вр)
  * [Общий вывод](#общий-вывод)



### Основные темы
|раздел | [nixtla](https://nixtlaverse.nixtla.io/statsforecast/) | [FPPPY](https://otexts.com/fpppy/)|
|-------|-----------------------|------|
|TSA intro |- | [Chapter 1](https://otexts.com/fpppy/nbs/01-intro.html) |
|EDA |- | [Chapter 2](https://otexts.com/fpppy/nbs/02-graphics.html) |
|Анализ компонент| - | [3.1 - 3.5](https://otexts.com/fpppy/nbs/03-decomposition.html) |

### Дополнительная литература
* [Доп. из skforecast](https://cienciadedatos.net/documentos/py71-visualizing-time-series-data)
* [EDA demand forecasting](https://newdigitals.org/2024/02/17/retail-sales-store-item-demand-time-series-analysis-forecasting-autoeda-fb-prophet-sarimax-model-tuning/)

### Импорт

#### Pandas

Проведем импорт необходимых библиотек

In [47]:
import pandas as pd 

#### Библиотеки для работы с датами

In [48]:
# !pip install holidays astral

In [49]:
import holidays
import astral

#### Базовые библиотеки

In [50]:
# !pip install seaborn plotly

In [51]:
import os
import numpy as np
import sklearn
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
from statsmodels.tsa.seasonal import seasonal_decompose

#### Работа с визуализацией

In [52]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates#Date Parser
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
sns.set_style('white')
sns.set(rc={'figure.figsize':(11, 4)})

#### Подовление некоторых варнингов работы с библиотеками

In [53]:
import warnings
#предупреждение  печати statsmodels (verbose)
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

## Загрузка данных в Pandas

### Чтение данных

#### открываем датасет

урок вдохновлён данным репозиторием https://github.com/jenfly/opsd и проектом https://www.machinelearningplus.com/time-series/time-series-analysis-python/

В качестве  набор данных для практики рассмотрим часть набор данных [открытые данные энергетических систем](https://open-power-system-data.org/). Мы будем работать с данным относящимися ко временным рядам (https://data.open-power-system-data.org/time_series/). По приведенной ссылке можно найти описание набора данных. Также набор можно найти на странице официального репозитория: https://github.com/Open-Power-System-Data/time_series.

доступные кода стран:
```
['AT', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']
```

Из всего набора выделим только данные относящиеся к целевой стране.  В качестве примера рассмотрим германию. В качестве периода анализа возьмём период с 2015 по 2019 годы.

Нам понадобятся не все колонки, поэтому выделим необходимые, кроме того приведем колонки к более интерпретируемому виду

Данные можно прочесть в отдельно файле `read_the_data.csv`

In [54]:
path_ts = 'de_hourly_power_and_weather.csv'
path_ts = os.path.join('utilits4tsc',path_ts)
df = pd.read_csv(path_ts)

#### первичные проверки

In [55]:
print(f'монтонность: {df.index.is_monotonic_increasing}')
print(f'дубликаты: {df.index.has_duplicates}') 
print(f'пропуски: {df.isna().sum().sum()}') 

монтонность: True
дубликаты: False
пропуски: 282


In [56]:
print(f'информация')
df.info()

информация
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   utc_timestamp                 43824 non-null  object 
 1   Consumption                   43824 non-null  float64
 2   Wind                          43750 non-null  float64
 3   Solar                         43721 non-null  float64
 4   Wind+Solar                    43719 non-null  float64
 5   temperature                   43824 non-null  float64
 6   radiation_direct_horizontal   43824 non-null  float64
 7   radiation_diffuse_horizontal  43824 non-null  float64
dtypes: float64(7), object(1)
memory usage: 2.7+ MB


### Описание данных

In [57]:
df.sample(5, random_state=0)

,utc_timestamp,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
7895,2015-11-25 23:00:00+00:00,51.775,6.814,0.000,6.814,1.761,0.0000,0.0000
26721,2018-01-18 09:00:00+00:00,75.599,35.258,1.486,36.744,3.710,2.1914,65.3990
26929,2018-01-27 01:00:00+00:00,45.762,2.611,0.000,2.611,2.888,0.0000,0.0000
42497,2019-11-06 17:00:00+00:00,69.906,3.438,0.000,3.438,6.019,0.0000,0.0000
38404,2019-05-20 04:00:00+00:00,54.850,5.199,1.169,6.368,11.206,5.9545,48.6132


Теперь набор представляет   потребление электричества (в гига-Ваттах в час) в Германии. Набор включает следующие временные ряды в виде колонок:

__Временная метка__
* `utc_timestamp` — дата в формате `гггг-мм-дд-чч-мм-сс + GMT`;
  
__Основные ряды__
* `Consumption` — Общее потребление, `ГВт/ч`;
* `Wind` — Потребление ветряной энергии, `ГВт/ч`;
* `Solar` — Потребление солнечной энергии, `ГВт/ч`;
* `Wind+Solar` — Потребление энергии из альтернативных источников, `ГВт/ч`.
  
__Экзогенные ряды__
* `temperature` - температура каждый день
* другие ряды 



### Введение индексов-дат

####  Задание временной индексации

1. **Переименование колонки**:  
   Колонка `'utc_timestamp'` переименована в `'time'` для удобства и читаемости.

2. **Преобразование к типу datetime**:  
   Преобразовали `'time'` к типу временных меток `datetime`. Это необходимо для корректной работы с временными рядами.

3. **Установка временного индекса**:  
   Колонка `'time'` установлена в качестве индекса DataFrame. Это позволяет использовать удобные методы для работы с временными данными, например:  
   - `df.loc['2018']` — выборка за 2018 год,  
   - `df.resample('D')` — агрегация по дням.
     
4. **Удаление часового пояса (UTC)**:  
   Из индекса удалён часовой пояс (например, `+00:00`), чтобы получить "наивное" время (`naive datetime`) в формате `2015-11-25 23:00:00`.  

   
   
5. **Проверка частоты данных**:  
   Убедимся, что временной индекс имеет **почасовую частоту** (`'H'`).
        
6. **Проверка**:  
   Вывод случайных 15 строк для оценки структуры данных.

In [58]:
df = df.rename(columns = {'utc_timestamp':'time'})
df['time'] = pd.to_datetime(df['time'])
df.set_index('time', inplace=True)
df.index = df.index.tz_localize(None)
df = df.asfreq('h')

df.sample(5, random_state=0)

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
time,,,,,,,
2015-11-25 23:00:00,51.775,6.814,0.000,6.814,1.761,0.0000,0.0000
2018-01-18 09:00:00,75.599,35.258,1.486,36.744,3.710,2.1914,65.3990
2018-01-27 01:00:00,45.762,2.611,0.000,2.611,2.888,0.0000,0.0000
2019-11-06 17:00:00,69.906,3.438,0.000,3.438,6.019,0.0000,0.0000
2019-05-20 04:00:00,54.850,5.199,1.169,6.368,11.206,5.9545,48.6132


#### проверка регулярности шагов временной метки

In [59]:
dt = df.index.to_series().diff().value_counts()
print(dt.head())

time
0 days 01:00:00    43823
Name: count, dtype: int64


#### чтение с временными метками

Отметим, что на самом деле можно было сразу загрузить данные в таком виде, чтобы индексы были датами

In [60]:
df = pd.read_csv(path_ts, parse_dates=['utc_timestamp'], index_col="utc_timestamp")
df.index.name = 'time'
df.index = df.index.tz_localize(None)
df = df.asfreq('h')
df.sample(15, random_state=0)

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
time,,,,,,,
2015-11-25 23:00:00,51.775,6.814,0.000,6.814,1.761,0.0000,0.0000
2018-01-18 09:00:00,75.599,35.258,1.486,36.744,3.710,2.1914,65.3990
2018-01-27 01:00:00,45.762,2.611,0.000,2.611,2.888,0.0000,0.0000
2019-11-06 17:00:00,69.906,3.438,0.000,3.438,6.019,0.0000,0.0000
2019-05-20 04:00:00,54.850,5.199,1.169,6.368,11.206,5.9545,48.6132
2018-03-13 18:00:00,71.345,16.488,0.000,16.488,3.767,0.0000,0.0000
2016-03-27 20:00:00,46.519,13.943,0.000,13.943,5.229,0.0000,0.0000
2018-03-17 17:00:00,63.897,36.804,0.022,36.826,-2.430,0.0477,1.4213
2019-05-09 03:00:00,51.400,15.056,0.022,15.078,7.077,0.0067,0.3814


### Работа с индексами

#### Даты


Данные обработанные в форме дат `DateTimeIndex` позволяют работать с индексом как с датой


In [61]:
print(df.index.day)
print(df.index.weekday)
print(df.index.year)

Index([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
       ...
       31, 31, 31, 31, 31, 31, 31, 31, 31, 31],
      dtype='int32', name='time', length=43824)
Index([3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       ...
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
      dtype='int32', name='time', length=43824)
Index([2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015, 2015,
       ...
       2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019, 2019],
      dtype='int32', name='time', length=43824)


#### обращение по содержанию индекса

Теперь мы можем обращаться к данным по дате

In [62]:
df.loc['2017-08-10']

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
time,,,,,,,
2017-08-10 00:00:00,42.716,5.050,0.000,5.050,14.825,0.0000,0.0000
2017-08-10 01:00:00,42.529,4.861,0.000,4.861,14.498,0.0000,0.0000
2017-08-10 02:00:00,42.991,4.332,0.000,4.332,14.181,0.0000,0.0000
2017-08-10 03:00:00,46.532,3.959,0.003,3.962,13.851,0.0029,0.1505
2017-08-10 04:00:00,53.360,3.609,0.253,3.862,13.878,0.9572,20.8054
2017-08-10 05:00:00,58.807,2.987,1.540,4.527,14.663,6.7372,80.7418
2017-08-10 06:00:00,61.960,2.653,3.428,6.081,15.508,20.7306,154.9284
2017-08-10 07:00:00,63.459,2.352,4.954,7.306,16.552,42.5766,226.9318
2017-08-10 08:00:00,64.522,2.256,6.706,8.962,17.570,68.3593,287.0262


##### срез по индексам (датам)

In [63]:
df.loc['2016-12-31 00:00':'2016-12-31 23:00']

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
time,,,,,,,
2016-12-31 00:00:00,42.871,11.799,0.000,11.799,-2.643,0.0000,0.0000
2016-12-31 01:00:00,41.751,11.696,0.000,11.696,-2.737,0.0000,0.0000
2016-12-31 02:00:00,40.998,11.707,0.000,11.707,-2.826,0.0000,0.0000
2016-12-31 03:00:00,40.881,11.587,0.000,11.587,-2.918,0.0000,0.0000
2016-12-31 04:00:00,40.731,11.699,0.000,11.699,-3.027,0.0000,0.0000
2016-12-31 05:00:00,40.594,12.179,0.000,12.179,-3.134,0.0000,0.0000
2016-12-31 06:00:00,42.833,12.408,0.001,12.409,-3.240,0.0000,0.0000
2016-12-31 07:00:00,46.007,12.941,0.214,13.155,-3.214,1.8404,8.9410
2016-12-31 08:00:00,49.248,13.026,2.147,15.173,-2.326,27.1864,52.2721


##### Обращение к колонке

In [64]:
df.loc['2015-12-20':'2015-12-25', 'Wind']

time
2015-12-20 00:00:00    13.676
2015-12-20 01:00:00    13.726
2015-12-20 02:00:00    13.919
2015-12-20 03:00:00    15.169
2015-12-20 04:00:00    16.171
                        ...  
2015-12-25 19:00:00    17.562
2015-12-25 20:00:00    18.847
2015-12-25 21:00:00    20.540
2015-12-25 22:00:00    23.037
2015-12-25 23:00:00    24.894
Freq: H, Name: Wind, Length: 144, dtype: float64

а также к каждой колонке можно обращаться по ключу

In [65]:
df[['Wind']].loc['2015-12-20':'2015-12-25']

,Wind
time,
2015-12-20 00:00:00,13.676
2015-12-20 01:00:00,13.726
2015-12-20 02:00:00,13.919
2015-12-20 03:00:00,15.169
2015-12-20 04:00:00,16.171
...,...
2015-12-25 19:00:00,17.562
2015-12-25 20:00:00,18.847
2015-12-25 21:00:00,20.540


##### Колонка как переменная

In [66]:
df.Wind.loc['2015-12-20':'2015-12-25']

time
2015-12-20 00:00:00    13.676
2015-12-20 01:00:00    13.726
2015-12-20 02:00:00    13.919
2015-12-20 03:00:00    15.169
2015-12-20 04:00:00    16.171
                        ...  
2015-12-25 19:00:00    17.562
2015-12-25 20:00:00    18.847
2015-12-25 21:00:00    20.540
2015-12-25 22:00:00    23.037
2015-12-25 23:00:00    24.894
Freq: H, Name: Wind, Length: 144, dtype: float64

##### Обращение по номеру индекса

также можно обращаться по индексу чрез метод iloc

In [67]:
df.iloc[0:2,0:3]

,Consumption,Wind,Solar
time,,,
2015-01-01 00:00:00,41.151,8.852,NaN
2015-01-01 01:00:00,40.135,9.054,NaN


#### Групперовка по датам

##### asfreq

Данные можно представлять с нужной частотой при помощи метода `asfreq`, например с частотой `D` - день, `W`,`M`,`Y` для недели, месяца и года соответственно.

In [68]:
df[['Wind']].loc['2015-10-20':'2015-12-25'].asfreq('W')

,Wind
time,
2015-10-25,7.441
2015-11-01,7.414
2015-11-08,25.551
2015-11-15,17.585
2015-11-22,10.280
2015-11-29,29.184
2015-12-06,26.210
2015-12-13,25.333
2015-12-20,13.676


можно сделать обращение по месяцу с периодом 1 неделя

In [69]:
df.loc['2016-02'].asfreq('W')

,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
time,,,,,,,
2016-02-07,42.550,21.905,0.0,21.905,3.770,0.0,0.0
2016-02-14,43.142,14.387,0.0,14.387,1.675,0.0,0.0
2016-02-21,43.698,28.060,0.0,28.060,6.184,0.0,0.0
2016-02-28,42.962,5.712,0.0,5.712,-2.771,0.0,0.0


или точно также с использованием открытой нотации обращения к массиву

In [70]:
df.loc['2015':].asfreq('YE')

ValueError: Invalid frequency: YE

Полагаем, что в данном случае будет визуально правильней поменять индексы на значение года

In [ ]:
df.loc['2012':].asfreq('YE').set_index(df.loc['2012':].asfreq('YE').index.year)

#### groupby

Для индексов в формате дат также доступно группирование  методом ``` groupby```. Группирование ``` groupby``` происходит по заданным периодам, например `W`, 'Y' или `A` (год), `'2y'` (по 2 года) и т.д.


Часто после использования методов  ``` groupby```, ```asfreq```, а также ``` groupby``` используется некоторая функция итога, например,  `sum`, `mean`, `median` or `std`.  

In [ ]:
df.groupby(pd.Grouper(freq='1YE')).sum()

#### resample

In [ ]:
df.resample('1W').median().head(3)

In [ ]:
df.asfreq('1W').head(3)

The full list of frequencies with its description can be find in this book
https://jakevdp.github.io/PythonDataScienceHandbook/03.11-working-with-time-series.html


## Первичный анализ данных 

### Описание датасета
Проведем анализ сформированного набора данных

__1. Общая информация о данных__

- **Форма данных**: `(количество строк, количество колонок)`
- **Индекс**: `имя индекса (тип данных)`
- **Диапазон дат**: `минимальная дата — максимальная дата`
- **Частота (если установлена)**: `H` (почасовая) или `None`

---

__2. Типы данных__

- **Распределение по типам**:  
  - `float64`: N колонок  
  - `int64`: M колонок  
  - и т.д.

---

__3. Пропущенные значения__

- **Количество колонок с пропусками**: K  
- **Топ-10 колонок с наибольшим числом пропусков**:  
  - `колонка_1`: N пропусков  
  - `колонка_2`: M пропусков  
  - и т.д.

> Если пропусков нет — выводится:  
> `Нет пропущенных значений.`

---

__4. Базовая статистика__

- **Числовые колонки**: описательная статистика (среднее, std, min, max, квантили).

In [ ]:
print(f"Описание: ")
print("-" * 50)
print(f"Форма данных: {df.shape}")
print(f"Индекс: {df.index.name} (тип: {df.index.dtype})")
print(f"Диапазон дат: {df.index.min()} — {df.index.max()}")
print(f"Частота (если установлена): {df.index.freq}")
print("-" * 50)    
print("\nТипы данных:")
print(df.dtypes.value_counts())
print("-" * 50)
print("\nПропуски:")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) == 0:
    print("Нет пропущенных значений.")
else:
    print(missing.head(10))  # топ-10 колонок с пропусками
print("-" * 50)
print("\nБазовая статистика (числовые колонки):")
df.describe()

Видно , что есть пропуски, в целом все ожидаемо. Посмотрим как распределены пропущенные значения

### Карта пропущенных значений

- **Цель**: визуализировать, в какие дни и по каким признакам имеются **пропущенные значения** в данных.
- **Метод**: 
  - Данные ресемплированы по дням (`resample('D')`).
  - Для каждого дня проверяется, есть ли хотя бы одно значение `NaN` по любому из признаков.
  - Результат транспонируется: строки — признаки, столбцы — дни.
- **Цвета**:
  - `Тёмный`: нет пропусков (все значения заполнены).
  - `Светлый`: есть хотя бы одно пропущенное значение в этот день.
- **Подписи оси X**: отображаются **раз в 10 недель** для лучшей читаемости.

In [ ]:

# Ресемплируем данные по дням, проверяя, есть ли хотя бы один пропуск в день
df_resampled = df.resample('D').apply(lambda x: x.isna().any())

# Транспонируем для визуализации
df_plot = df_resampled.T
df_plot.columns = df_plot.columns.date

plt.figure(figsize=(18, 6))
sns.heatmap(df_plot, cbar=True, cmap='viridis', yticklabels=True)
plt.title('Карта пропущенных значений (по дням, ресемплировано)')

# Устанавливаем шаг по датам (например, каждые 10 недель)
step = 7 * 10  # каждые 10 недель
xticks_pos = range(0, len(df_plot.columns), step)
xticks_labels = [df_plot.columns[i] for i in xticks_pos]
plt.xticks(ticks=xticks_pos, labels=xticks_labels, rotation=45)

plt.tight_layout()
plt.show()

В наборе достаточно много значений `NaN`. 

В данном случае представляется интересной задача восстоновления их значений по данным температуры и общего потребления. Однако если это неудается, то  в `Pandas` есть несколько инструментов, в том числе  ```ffill```, ```bfill``` для заполнения пропусков соответственно следующими или предыдущими значениями, например можно использовать  метод так ```.asfreq('D', method='ffill')```. Также возможны использования методов удаления пропусков `dropna` или заполнения заданными значениями `filna`. Если используется метод `dropna`, то из данных будет удалена вся строка с пропуском.
    
Давайте для начала посмотрим сколько у нас `NaN`  значений. 
Для этого можно использовать или метод `isnull` или `isna`
.

In [ ]:
df.isna().sum()

In [ ]:
df = df.ffill().bfill()
df.head(3)

### Создание признаков

В наших данных есть составляющее общего потребления, солнечной и ветряной энергии. Полагаем, что нам может потребоваться столбец, соответствующий другим источникам (не альтернативным).

In [ ]:
df['Traditional'] = df['Consumption'] - df['Solar'] - df['Wind'] 

## Визуализация временного ряда


### Pandas визуализация 

#### Plot переменной

In [ ]:
df['Consumption'].plot(linewidth=0.5);

#### Настройка плота

Также давайте попробуем визуализировать и остальные столбцы

In [ ]:
cols_plot = ['Consumption', 'Solar', 'Wind','Traditional','temperature']
axes = df[cols_plot].plot(marker='.', alpha=0.4, linestyle='-', figsize=(11, 9), subplots=True)
# for ax,col in axes:
#     ax.set_ylabel('Daily Totals (GWh)')

### Визуализация на одном графике, с периодом 1 месяц относительно экзогенного фактора - температура

In [ ]:
# Агрегация по месяцам: сумма для энергии, среднее для температуры
df_monthly_energy = df[['Consumption', 'Wind', 'Solar']].resample('ME').sum(min_count=1)
df_monthly_temp = df[['temperature']].resample('ME').mean()  # среднее значение температуры за месяц
df_monthly = df_monthly_energy.join(df_monthly_temp)

fig, ax1 = plt.subplots(figsize=(14, 4))

# Основной график: потребление
ax1.plot(df_monthly['Consumption'], color='black', label='Total Consumption')
ax1.set_ylabel('Monthly Total (GWh)', color='black')
ax1.tick_params(axis='y', labelcolor='black')

# Область: генерация ВИЭ
df_monthly[['Wind', 'Solar']].plot.area(ax=ax1, linewidth=0, alpha=0.6)

# Вторая ось Y для температуры
ax2 = ax1.twinx()
ax2.plot(df_monthly['temperature'], color='red', linestyle='--', label='Avg Temperature')
ax2.set_ylabel('Temperature (°C)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Форматирование оси X
ax1.xaxis.set_major_locator(mdates.YearLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Легенда
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Monthly Energy Consumption and Weather (Temperature)')
plt.tight_layout()
plt.show()

#### Анализ полученных графиков показывает следующее:
* Все три графика осциллируют во времени в течение года. Рискнем предположить, что это связано с сезонностью и изменением погоды. 
* Потребление электроэнергии выше зимой и ниже летом.
* Потребление энергии имеет 2 сезонных составляющих:
  * основная, с  примерным диапазоном значений 1300-1500 ГВт 
  * дополнительная со значениями порядка 1100 ГВт, которая предположительно связана с изменением потребления в течение неделе. Это также подтверждается заметным снижением потребления в начале каждого года. 
* Пик производства солнечной энергии приходится на лето.
* Пик производства ветряной энергии приходится на зиму, причем колебания этого ряда куда более подвержены дисперсии. Полагаем, что это связано с погодным фактором.
* Значение альтернативных источников энергии растет, но очень медленно.
* Общее электропотребление, а также потребление из альтернативных источников имеют растущий тренд, тогда как тренд традиционных источников - спадающий.




### Интерактивная визуализация

In [ ]:
# Подготовка данных
df_viz = df[['Consumption', 'temperature']].copy()
df_viz['Consumption_rolling_7d'] = df_viz['Consumption'].rolling(24*7, min_periods=1).mean()

# Создаём подграфики с двумя осями Y
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Правая ось: температура
fig.add_trace(
    go.Scatter(x=df_viz.index, y=df_viz['temperature'], name='Temperature', 
               line=dict(color='red', dash='dash'), opacity=0.7),
    secondary_y=True,
)

# Левая ось: потребление
fig.add_trace(
    go.Scatter(x=df_viz.index, y=df_viz['Consumption'], name='Actual Consumption', 
               line=dict(color='blue', width=1), opacity=0.97),
    secondary_y=False,
)

# Левая ось: скользящее среднее
fig.add_trace(
    go.Scatter(x=df_viz.index, y=df_viz['Consumption_rolling_7d'], name='7-Day Rolling Mean', 
               line=dict(color='black', dash='dot', width=3), opacity=0.99),
    secondary_y=False,
)

# Настройка осей
fig.update_layout(title="Consumption with Rolling Mean & Temperature (Dual Axis)")
fig.update_xaxes(title_text="Time")
fig.update_yaxes(title_text="GWh", secondary_y=False, side='left')
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True, side='right')

fig.show()

### Оценка виляния температуры на показания

##### Кажется очевидной гипотеза, что температура, время суток и время года обратно пропорциональны потреблению, давайте это проверим

In [ ]:

# Подготовка данных
df['temp_bin'] = pd.cut(df['temperature'], bins=20)
df['hour'] = df.index.hour
df['month'] = df.index.month

# Данные для двух графиков
heatmap_hour = df.groupby(['temp_bin', 'hour'])['Consumption'].mean().unstack(fill_value=0)
heatmap_month = df.groupby(['temp_bin', 'month'])['Consumption'].mean().unstack(fill_value=0)

# Создаём 2 сабплота горизонтально с общей осью Y и общей цветовой шкалой
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6), sharey=True)

# График 1: по часам
sns.heatmap(heatmap_hour, cmap='viridis',  ax=ax1, cbar=False)
ax1.set_title('Average Consumption by Temperature Range and Hour')
ax1.set_ylabel('Temperature Range')
ax1.set_xlabel('Hour of Day')

# График 2: по месяцам
sns.heatmap(heatmap_month, cmap='viridis',  ax=ax2, cbar_kws={'label': 'Consumption (GWh)', 'shrink': 0.8})
ax2.set_title('Average Consumption by Temperature Range and Month')
ax2.set_ylabel('')  # Убираем подпись оси Y для второго графика
ax2.set_xlabel('Month')

plt.tight_layout()
plt.show()

##### Осталось понять как конкретно влияет температура на показания, есть ли там линейная зависимость например?

In [ ]:
 
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
import numpy as np

# Ресемплирование по неделям
df_weekly = df[['temperature', 'Consumption']].resample('W').mean().dropna()

# Подготовка данных
X = df_weekly[['temperature']].values
y = df_weekly['Consumption'].values

# Полиномиальные признаки (степень 2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Регрессия с регуляризацией (Ridge для полинома)
model = Ridge(alpha=1.0)
model.fit(X_poly, y)

# Предсказания
y_pred = model.predict(X_poly)

# Сортировка для плавной линии
sorted_indices = np.argsort(df_weekly['temperature'])
x_sorted = df_weekly['temperature'].iloc[sorted_indices]
y_sorted = y_pred[sorted_indices]

# Коэффициент корреляции
correlation = np.corrcoef(df_weekly['temperature'], df_weekly['Consumption'])[0, 1]

# Создание графика
fig = go.Figure()

# Скаттерплот
fig.add_trace(go.Scatter(
    x=df_weekly['temperature'],
    y=df_weekly['Consumption'],
    mode='markers',
    name='Weekly averages',
    marker=dict(size=8, opacity=0.6),
    text=df_weekly.index.strftime('%Y-%m-%d'),
    hovertemplate='<b>Date</b>: %{text}<br>' +
                  '<b>Temp</b>: %{x:.1f} °C<br>' +
                  '<b>Consumption</b>: %{y:.2f} GWh<extra></extra>'
))

# Полиномиальная регрессия
fig.add_trace(go.Scatter(
    x=x_sorted,
    y=y_sorted,
    mode='lines',
    name='Polynomial Regression (degree=2)',
    line=dict(color='red'),
    hovertemplate='Poly fit: %{y:.2f} GWh<extra></extra>',
))

# Настройка графика
fig.update_layout(
    title=f'Weekly Average Temperature vs Consumption with Polynomial Regression (Corr: {correlation:.3f})',
    xaxis_title='Temperature (°C)',
    yaxis_title='Consumption (GWh)',
    hovermode='closest'
)

fig.update_layout(
    width=1000,
    height=600
) 

fig.show()

# Вывод коэффициентов
print(f"Polynomial Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_:.3f}")
print(f"Correlation coefficient: {correlation:.3f}")

##### Видно, что на недельном масштабе меет место близкая к полиномиальной зависимость. Однако, есть несколько отклонений для периода конец декабря - начала января, а также для апреля 2015 года. 

Однако, также важно понять как быстро происходит реакция потребления на температуру. Для этого можно посчитать корреляцию со сдвинутой копией ряда потребления. То есть лаговую корреляцию. Максимум по модулю этой величины покажет когда и насколько хорошо потребление реагирует на изменение температуры

Важно понимать, что корреляция по лагам не причинность и может быть:
* завышена сезонностью
* смазана суточным циклом

Код ниже

In [ ]:
df['day_of_week'] = pd.to_datetime(df.index).dayofweek
# Дедтренд: убираем сезон и суточный цикл
df['Consumption_detrended'] = df['Consumption'] - df.groupby(['month', 'hour'])['Consumption'].transform('mean')

# Определяем сезоны
season_map = {1: 'Winter', 2: 'Spring', 3: 'Summer', 4: 'Autumn'}
df['season'] = df['month'].map(lambda m: (m%12 + 3)//3).map(season_map)

# Подписи дней недели
dow_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df['day_of_week_str'] = df['day_of_week'].map(dow_map)

# Параметры лагов
max_lag = 13
lags = range(0, max_lag + 1)

results = []

# Вычисляем корреляцию по сезону и дню недели
for season in df['season'].unique():
    for dow in df['day_of_week_str'].unique():
        sub = df[(df['season'] == season) & (df['day_of_week_str'] == dow)]
        if len(sub) < 30:
            continue
        
        corrs = [sub['Consumption_detrended'].corr(sub['temperature'].shift(lag)) for lag in lags]
        corrs_smoothed = pd.Series(corrs).rolling(3, center=True, min_periods=1).mean()
        
        best_lag = corrs_smoothed.abs().idxmax()
        best_corr = corrs_smoothed[best_lag]
        
        results.append({
            'season': season,
            'day_of_week': dow,
            'best_lag': best_lag,
            'corr': best_corr
        })

best_lags_df = pd.DataFrame(results)


# Heatmap: лучший лаг
pivot_lag = best_lags_df.pivot(index='day_of_week', columns='season', values='best_lag')
sns.heatmap(pivot_lag, annot=True, cmap='coolwarm', fmt='.0f')
plt.title('Best temperature lag (hours) — detrended consumption')
plt.xlabel('Season')
plt.ylabel('Day of week')
plt.show()

# Heatmap: сила корреляции
pivot_corr = best_lags_df.pivot(index='day_of_week', columns='season', values='corr')
sns.heatmap(pivot_corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation at best lag — detrended consumption')
plt.xlabel('Season')
plt.ylabel('Day of week')
plt.show()

##### Видно, что чаще потребление элеткроэнергии догоняет изменение температуры через 9-13 часов. Лучше всего смотреть на корреляцию в воскресенье, когда меньше мешающих факторов

## Оценка сезонных составляющих

### Поиск сезонных составляющих

Давайте подробней изучить сезонность по подробнее.
Для анализа гипотезы о наличии двух составляющих в общем потреблении давайте посмотрим на распределение значений.

In [ ]:
sns.histplot(df['Consumption'], kde=True, stat='density', bins=100)
plt.title('Density of Consumption (with 2 peaks)')
plt.xlabel('Consumption (GWh)')
plt.ylabel('Density')
plt.show()

Предположительно сезонность годовая и суточная - это можно визуализировать. Однако будем проверять больше гипотез:
* Начнем с месячного масштаба. Цель: увидеть, в какие месяцы потребление высокое/низкое → зима vs лето.
* Также посмотрим на часовой масшта. Цель: понять, в какое время суток пик нагрузки → утро и вечер (типичный "пик потребления").
* Проверим также масштаб дней недели. Цель: понять, в какие дни  пик нагрузки → будни vs выходные.

In [ ]:
# Подготовка данных
df['month'] = df.index.month
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek  # 0=понедельник, 6=воскресенье

# Создаём 3 сабплота горизонтально
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

# График 1: по месяцам
sns.boxplot(x='month', y='Consumption', data=df, ax=ax1)
ax1.set_title('Distribution of Consumption by Month')
ax1.set_xlabel('Month')
ax1.set_ylabel('Consumption (GWh)')

# График 2: по часам
sns.boxplot(x='hour', y='Consumption', data=df, ax=ax2)
ax2.set_title('Distribution of Consumption by Hour of Day')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Consumption (GWh)')

# График 3: по дням недели
sns.boxplot(x='day_of_week', y='Consumption', data=df, ax=ax3)
ax3.set_title('Distribution of Consumption by Day of Week')
ax3.set_xlabel('Day of Week')
ax3.set_ylabel('Consumption (GWh)')
ax3.set_xticks(range(7))
ax3.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.tight_layout()
plt.show()

Дополнительно проверим вполне очевидную гипотезу, что чем меньше световой день, тем выше потребление

In [ ]:
from astral import LocationInfo
from astral.sun import sun

city = LocationInfo("Berlin", "Germany", timezone="Europe/Berlin")

def daylight_hours(ts):
    s = sun(city.observer, date=ts.date())
    return (s['sunset'] - s['sunrise']).total_seconds() / 3600

df['daylight_duration'] = df.index.map(daylight_hours)


fig = px.scatter(
    df, x='daylight_duration', y='Consumption',
    hover_data={df.index.name or 'index': df.index.astype(str)},
    title='Consumption vs Daylight Duration',
    width = 1000)

fig.show()


Видно, что поведение довольно очевидное, однако имеют место аномалии в декабре-январе и в мае 2015 и в сентябре 2016 года.

Также проверим что действительно потребление в дневное время выше, чем в ночное по месяцам

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)

pivot = df.pivot_table(
    values='Consumption',
    index='hour',
    columns='month',
    aggfunc='mean'
)

sns.heatmap(pivot, cmap='viridis', ax = axes[0])



pivot_h_dow = df.pivot_table(
    values='Consumption',
    index='hour',
    columns='day_of_week',
    aggfunc='mean'
)

sns.heatmap(pivot_h_dow, cmap='viridis', ax = axes[1])

Внутрисуточная сезонность как и недельная и месячная очевидна

### Анализ редких регулярных событий

Из анализа видно, что также имеет место странное поведение в период декабрь - январь, поведение уже было ранее выявлено. Нужно проверить его подробнее

In [ ]:
# Подготовка данных
df['week'] = df.index.isocalendar().week  # номер недели в году

# Создаём 2 сабплота горизонтально
fig, ax1 = plt.subplots(1, 1, figsize=(16, 4))


# График 2: по неделям
sns.boxplot(x='week', y='Consumption', data=df, ax=ax1)
ax1.set_title('Distribution of Consumption by Week')
ax1.set_xlabel('Week of Year')
ax1.set_ylabel('Consumption (GWh)')

plt.show()

Спад потребления связан с выходными. Однако, Потребление во время смены года можно считать или аномалией или третьей составляющей (что возможно правильней). Проверим гипотезу о 3 составляющей. Для этого используем скользящее среднее.

In [ ]:
ax = df.loc['2013-10':'2017-05', 'Consumption'].\
    resample('W').mean().plot(marker='o', linestyle='-',linewidth=1.5)
ax.set_ylabel('Daily Consumption (GWh)')
plt.show()

В целом, подтвердить гипотезу о holidays можно и по другому, например при помощи библиотеки `holidays`

In [ ]:
import holidays

# Пример для Германии
de_holidays = holidays.DE(years=df.index.year.unique())

# Используем .apply() для Series или .map() для Series
df['is_holiday'] = df.index.to_series().dt.date.apply(lambda x: x in de_holidays)

# Визуализация
sns.boxplot(data=df, x='is_holiday', y='Consumption')
plt.title('Consumption on Holidays vs Regular Days')
plt.show()

И так, у нас имеет место 2 основных сезонности - годовая и недельная и регулярное но редкое событие - окончание каждого года.

### Оценка тренда

Предположительно потребление растет, однако, это необходимо проверить. Для этого воспользуемся продвинутым аналогом функции скользящее среднее.

In [ ]:
decomp = seasonal_decompose(df['Consumption'], model='additive', period=24*365//1)  # 1 год = 8760 часов
decomp.plot()
plt.tight_layout()
plt.show()

Тренд довольно небольшой и возможно, что циклический. Посмотрим также на тренд в разрезе лет

In [ ]:
ax=sns.violinplot(data=df, x=df.index.year, y='Consumption',
               split=True, inner="quart", linewidth=1, )
ax.set_ylabel('GWh')
ax.set_xlabel('year')
ax.set_title('Consumption per year')
plt.show()

## Анализ многопеременного ВР

### Объявление перепенных

Важно понимать когда нужно использование многопеременных ВР. Такие системы анализируют совместно если 
- учет лагов одной переменной стабильно влияет (улучшает прогноз) друго переменной.
- имеет место статистическая связь перепенных (казуальная или корреляционная).
- есть физические основания (в предметной области) считать систему действующей совместно.
- моделирование большого числа однопеременных ВР вычислительно сложная задача.
- 
В ряде практических случаев ограничиваются однопеременными ВР - так как это проще и дает удволитворяющий результат.

Аналогичная ситуация сохарняется и для управляющих (экзогенных) ВР, если они влияют на всю систему.

Лучше всего проверить необходимость мультипеременных ВР на практике. Однако, есть и априорные тесты. Наиболее важные из таких тестов это анализ корреляции переменных и анализ их казуальности.
* Если корреляция показывает именно статистическую связь пермерных, то казуальность говорит о статистической (не физической) причино-следственной связи. То есть казуальность - это изменение X при фиксированных остальных условиях приводит к изменению Y (как правило связь одностороняя). Ометим, что если известно что есть физическая связь между пеменными, то такой факт сильнее казуальной связи.

In [ ]:
system_cols = ['Consumption', 'Wind', 'Solar', 'Wind+Solar', 'Traditional']
ex_cols = ['temperature', 'radiation_direct_horizontal', 'radiation_diffuse_horizontal']

df_ = df[system_cols+ex_cols].copy()


#### Учет лагов экзогенного фактора

Мы уже выдвигали гипотезу, что температура воздуха влияет на потребление с заданным лагом (временной задержкой). Проверим этот фактор еще раз, задав задержанную копию этого ВР.

In [ ]:
lags = 13
df_[f'temperature_{lags}lags'] = df_['temperature'].shift(-lags).ffill().bfill()
df_.index = pd.to_datetime(df_.index)
ex_cols +=  [f'temperature_{lags}lags']

### Удаление празднечных дней

Редкие но регулярные события учитвать сложно при анализе переменных. Если мы считаем их эффект не значительным для всей системы - можно попробывать удалить их при анализе.

In [ ]:
de_holidays = holidays.Germany(
    years=range(df_.index.year.min(), df_.index.year.max() + 1),
    subdiv=None  # None = федеральные праздники
)

# Преобразуем в список дат (без времени)
holiday_dates = pd.to_datetime(list(de_holidays.keys())).normalize()

In [ ]:
extra_days = []
for year in range(df_.index.year.min(), df_.index.year.max() + 1):
    extra_days.extend([
        f'{year}-12-24',  # Сочельник
        f'{year}-12-31',  # Новогодний вечер
        f'{year}-01-01',  # Новогодний день
    ])
extra_dates = pd.to_datetime(extra_days).normalize()
holiday_dates = holiday_dates.union(extra_dates)

Заменим праздники значением того же дня на прошлой неделе!

In [ ]:
# Маска праздников (весь день)
holiday_mask = df_.index.normalize().isin(holiday_dates)

# Для каждой колонки делаем seasonal imputation: "тот же час недель назад"
for col in df_.columns:
    # Сдвигаем данные на 7 дней назад (168 часов для почасовых данных)
    shifted = df_[col].shift(freq='7D')  # ← именно '7D', не 168H, чтобы сохранить соответствие времени суток
    
    # Заменяем ТОЛЬКО праздничные значения на значения из прошлой недели
    df_.loc[holiday_mask, col] = shifted.reindex(df_.loc[holiday_mask].index).values

# Если остались NaN (например, праздник в первые 7 дней данных), заполним fallback-методом
df_ = df_.fillna(method='bfill').fillna(method='ffill')

### Разложение переменных на составляющие. 
Цель - получение остаточных частей ВР. 

Ометим, что Если у ВР может быть выделен понятный тренд и сезнность, то их предсказание является решенной задачей. Однако такие тренд и сезонность могут не полностью описывать ВР. Тогда следует заняться предсказанием для остаточной части. 

В нашем случае мы будем анализировать корреляционные свойства остаточной части, предполагая что тренд и сезонность могут дать ложные корреляции. Также важно понимать, что большая часть тестов расчитаны на т.н. стационарые ВР - то есть ВР для которых среднее и дисперсия сохраняются во времени. Но если есть основания учитывать не только остаток, то нужно, например, работать с ВР без тренда или иным способом.

При разложении будем учитывать что у нас есть несколько сезонностей. 
Так как у нас 3 составляющих в результате разложения получим полный тренд, полную сезонность и остаток от их разложения.

In [ ]:
# создадим словарь для хранения компонентов
decomposed = {}

# функция для многосезонного декомпозиции
def multi_seasonal_decompose(series, daily=24, weekly=168, yearly=8760):
    result = pd.DataFrame(index=series.index)
    
    # дневная сезонность
    daily_comp = seasonal_decompose(series, period=daily, model='additive', extrapolate_trend='freq')
    result['trend_daily'] = daily_comp.trend
    result['season_daily'] = daily_comp.seasonal
    result['resid_daily'] = daily_comp.resid
    
    # недельная сезонность на остатках после дневной
    weekly_comp = seasonal_decompose(series - daily_comp.seasonal, period=weekly, model='additive', extrapolate_trend='freq')
    result['trend_weekly'] = weekly_comp.trend
    result['season_weekly'] = weekly_comp.seasonal
    result['resid_weekly'] = weekly_comp.resid
    
    # годовая сезонность на остатках после дневной + недельной
    yearly_comp = seasonal_decompose(series - daily_comp.seasonal - weekly_comp.seasonal, period=yearly, model='additive', extrapolate_trend='freq')
    result['trend_yearly'] = yearly_comp.trend
    result['season_yearly'] = yearly_comp.seasonal
    result['resid_yearly'] = yearly_comp.resid
    
    return result

# применяем к системным колонкам
for col in system_cols+ex_cols:
    decomposed[col] = multi_seasonal_decompose(df_[col])
    df_[col + '_trend'] = decomposed[col]['trend_daily'] + decomposed[col]['trend_weekly'] + decomposed[col]['trend_yearly']
    df_[col + '_season'] = decomposed[col]['season_daily'] + decomposed[col]['season_weekly'] + decomposed[col]['season_yearly']
    df_[col + '_resid'] = decomposed[col]['resid_yearly']  # остатки после всех сезонностей
    df_[col + '_detrend'] = df_[col + '_season']+df_[col + '_resid']


### Анализ корреляции

Анализ корреляций  по остаткам (после удаления тренда и сезонности) —  рекомендуемый подход в контексте временных рядов в силу возможности появления ложных корреляций.
Ложные корреляции могут быть обсуловлены, например общим трендом (например, рост потребления или падение солнечной генерации зимой).
В этом смысле остатки - это то, что в системе нельзя объяснить общей динамикой.

In [ ]:
corr_system = df_[[f'{c}_resid' for c in system_cols]].corr()
plt.figure(figsize=(8, 4))
sns.heatmap(corr_system, annot=True, cmap='coolwarm', center=0)
plt.xticks(rotation=30, ha='right')  
plt.title('Корреляция остатков (residuals)')
plt.tight_layout()  # чтобы подписи не обрезались
plt.show()

результаты показывают, что wind + solar почти полностью определяются ветрянной энергией, при этом когда традиционные источники на максимуме, ветрянная энергия наоборот. Полное энергопотребление не зависит от других показателей.

показатели 'Wind+Solar', 'Traditional' можно далее не рассматривать.


In [ ]:
system_cols = [x for x in system_cols if not x in ['Wind+Solar', 'Traditional']]
 

Проведем также анализ казуальности - то есть анализ того, как задержанные версии переменных будут связаны с экзогенными переменными

In [ ]:
system_cols = ['Consumption', 'Wind', 'Solar']
ex_cols = ['temperature', 'radiation_direct_horizontal']

max_lag = 24
lags = np.arange(-max_lag, max_lag + 1)

fig, axes = plt.subplots(
    nrows=len(ex_cols),
    ncols=len(system_cols),
    figsize=(18, 8),
    sharey=True
)

for i, ex in enumerate(ex_cols):
    for j, sys in enumerate(system_cols):

        x = df_[f'{ex}_resid'].values
        y = df_[f'{sys}_resid'].values

        corrs = [
            np.corrcoef(
                np.roll(x, lag)[max_lag:-max_lag],
                y[max_lag:-max_lag]
            )[0, 1]
            for lag in lags
        ]

        ax = axes[i, j]
        ax.plot(lags, corrs, marker='o', linewidth=1)
        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)

        if i == 0:
            ax.set_title(sys, fontsize=12)

        if j == 0:
            ax.set_ylabel(f'{ex}\nCorrelation', fontsize=11)

        ax.set_xlabel('Lag (hours)')
        ax.grid(alpha=0.3)

plt.suptitle(
    'Lagged cross-correlation (residuals)\nRows: exogenous factors, Columns: system variables',
    fontsize=14
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


явная зависимость есть в основном у solar c незначительное задержкой. Однако, проведем также и тест на казуальность

### Granger causality tests

#### Повторная оценка влияния температуры на потребление

Данный тест часто применяется для анализа казуальности ВР. Для теста также нужны остатки (стационарные ВР). Такая казуальность покажет краткосрочную динамику системы (вплоть до 1 суток). 

Отметим, что иногда также анализируют казуальность тренда, чтобы оценить долгосрочные зависимости, однако это отдельный сложный вопрос.

In [ ]:
system_cols = ['Consumption', 'Wind', 'Solar']
ex_cols = ['temperature', 'radiation_direct_horizontal']

df_resid = df_[
        [f'{c}_resid' for c in system_cols] +
        [f'{c}_resid' for c in ex_cols]].dropna()

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
 
def granger_min_pvalue(df, cause, effect, max_lag=24):
    """
    Возвращает минимальный p-value по всем лагам и позицию лага
    """
    data = df[[effect, cause]].dropna()
    results = grangercausalitytests(data, maxlag=max_lag, verbose=False)
    pvals = [results[l][0]['ssr_ftest'][1] for l in results]
    return np.min(pvals), np.argmin(pvals)

In [ ]:
granger_min_pvalue(
    df_resid,
    cause='temperature_resid',
    effect='Consumption_resid',
    max_lag=24
)

p < 0.05 → лаги температуры улучшают прогноз потребления.

Также еще раз подтверждена гипотеза что температура дает эффект потребления в течении примерно 13 часов на горизонте до 24 часов.
 

#### Анализ экзогенных факторов

In [ ]:
causes = [f'{c}_resid' for c in ex_cols]
effects = [f'{c}_resid' for c in system_cols]
for cause in causes:
    for effect in effects:
        res = granger_min_pvalue(df_resid, cause=cause, effect=effect, max_lag=24  )
        print(f'{cause} -> {effect} | granger_pval:\t{res[0]:.3e}')

Видно что система имеет связи и некоторые из них, например radiation_direct_horizontal -> Solar достаточно сильныне.  Однако, это не говорит о фактическом эффекте, который будет проверен позже. Также не доконца ясна физическая обоснованность ряда связей, их учет сомнителен. 

In [ ]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Списки причин и следствий
causes = [f'{c}_resid' for c in ex_cols]
effects = [f'{c}_resid' for c in system_cols]

# Собираем результаты
results = []

for cause in causes:
    for effect in effects:
        res = granger_min_pvalue(df_resid, cause=cause, effect=effect, max_lag=24)
        pval, lag = res
        results.append({
            'cause': cause,
            'effect': effect,
            'pval': pval,
            'lag': lag
        })

# Создаём DataFrame
granger_df = pd.DataFrame(results)

# Вытаскиваем p-значения (игнорируя NaN)
pvals = granger_df['pval'].dropna()
original_idx = granger_df.dropna().index

# Применяем FDR-коррекцию (Benjamini-Hochberg)
_, pvals_corrected, _, _ = multipletests(pvals.values, alpha=0.05, method='fdr_bh')

# Добавляем откорректированные p-значения в DataFrame
granger_df.loc[original_idx, 'pval_corrected'] = pvals_corrected

# Фильтруем значимые результаты (после коррекции)
significant = granger_df[granger_df['pval_corrected'] < 0.05]

# Выводим результаты
if not significant.empty:
    print("Значимые Granger-причинности после FDR-коррекции (p < 0.05):")
    for _, row in significant.iterrows():
        print(f"{row['cause']} → {row['effect']} | p_corrected: {row['pval_corrected']:.3e}, lag: {row['lag']}")
else:
    print("Нет значимых Granger-причинностей после FDR-коррекции.")

## Общий вывод

Проанализированная система состоит из 3 основных переменных и 2 экзогенных факторов. В системе имеет место 3 основных сезонных составляющих: дневная, недельная и годовая. Также в системе есть регулярыные события по типу новогодних праздников.

сохраним получившийся датафрейм

In [ ]:
df = df.drop(columns = ['Wind+Solar','Traditional','radiation_diffuse_horizontal','temp_bin'],errors='ignore')
for col in  [x+ '_resid' for x in system_cols+ex_cols ]:
    df[col] = df_[col]

In [ ]:
df.to_csv('OPDS_DE_EDA.csv', index_label="time",  float_format="%.6f")

In [ ]:
df = pd.read_csv(
    "OPDS_DE_EDA.csv",
    parse_dates=["time"],
    index_col="time"
)
df.head(1)

### Упражнения
1. Выбрать один или несколько временных рядов с переменными и управляющими факторами, для которых определить:
   * Наличие тренда,
   * число и характер сезонных компонент,
   * наличие редких регулярных событий (например, праздники),
   * наличие и важность управляющих факторов
   * связанность переменных
   * необходимый временной шаг для анализа
   * другие особенности

In [ ]:
import session_info

session_info.show(html=False)

In [ ]:
session_info.show()